# Deep Learning with PyTorch

This notebook is a hands-on introduction to deep learning using PyTorch. We start from raw pixel data, build a neural network from scratch, understand every step of training, and learn how to diagnose and fix overfitting.

By the end of this notebook you will:
- Know how a neural network is built in PyTorch
- Understand every line of the training loop
- Be able to read loss curves and diagnose overfitting
- Know how dropout regularization works in practice

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

torch.manual_seed(42)

COLOR_BASELINE = "#4C72B0"  # blue
COLOR_DROPOUT  = "#DD8452"  # orange
DATA_DIR       = "./data"

## 1. Dataset: MNIST Handwritten Digits

MNIST contains 70,000 grayscale images (28×28 pixels) of handwritten digits 0–9.

We split the original 60,000 training examples into:
- **Train** (80% = 48,000): used to update the network's weights
- **Validation** (20% = 12,000): used to monitor generalization during training — the network never trains on these
- **Test** (10,000, held out): used exactly once, at the very end

The first thing you will see is a sample of images — get a feel for the task before any math.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),  # MNIST mean and std
])

full_train = datasets.MNIST(root=DATA_DIR, train=True,  download=True, transform=transform)
test_set   = datasets.MNIST(root=DATA_DIR, train=False, download=True, transform=transform)

n_train = int(0.8 * len(full_train))  # 48,000
n_val   = len(full_train) - n_train   # 12,000
train_set, val_set = random_split(
    full_train, [n_train, n_val], generator=torch.Generator().manual_seed(42)
)
print(f"Train: {len(train_set):,} | Val: {len(val_set):,} | Test: {len(test_set):,}")

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE)

print(f"Batches  train={len(train_loader)} | val={len(val_loader)} | test={len(test_loader)}")

In [ ]:
images_sample, labels_sample = next(iter(train_loader))

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    img = images_sample[i].squeeze() * 0.3081 + 0.1307  # unnormalize
    ax.imshow(img, cmap="gray")
    ax.set_title(str(labels_sample[i].item()), fontsize=12)
    ax.axis("off")
plt.suptitle("Sample images from the training set — label shown above each image", fontsize=12)
plt.tight_layout()
plt.show()

### What you just saw

The images are 28×28 pixels, the labels are clear (0–9), and the task is easy for a human. But before we build a model, we need to decide: what structure should the network have?

**Next question:** What does a neural network actually look like inside?

## 2. Building a Network from Scratch

### What `sklearn` was hiding

When you called `MLPClassifier` in scikit-learn, a neural network was built and trained behind a one-line API. Under the hood, three fundamental things were happening:

1. **Layers:** the network is organized in layers. Each layer applies a linear transformation: multiply the input vector by a weight matrix and add a bias vector.
2. **Weights:** each layer has a matrix of *weights* (and a bias vector). These are the parameters that training will adjust — they start random and improve over time.
3. **Activation functions:** after each linear layer (except the last), a non-linear function — we use ReLU — is applied. Without these, stacking multiple linear layers would collapse into a single linear transformation, and the network could only learn linear decision boundaries.

In PyTorch, we define all of this explicitly.

### Why we flatten: 28×28 → 784

Each MNIST image is a 2D grid of 28×28 = 784 pixels. A fully connected (`nn.Linear`) layer expects a **1D vector** as input. So before passing an image to the first layer, we flatten it: stack the 28 rows end-to-end to get a single vector of 784 numbers.

This works, but it discards all spatial relationships — pixel (0,0) and pixel (0,1) are neighbors in the image, but after flattening they are just two entries in a vector with no special connection. This is a fundamental limitation of fully connected networks on image data, which we will revisit at the end of this section.

In [ ]:
class FullyConnectedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()        # 28x28 -> 784
        self.fc1 = nn.Linear(784, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)  # raw logits — CrossEntropyLoss handles softmax

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

torch.manual_seed(42)
model = FullyConnectedNet().to(device)
print(model)
print()
print(f"Total trainable parameters: {sum(p.numel() for p in model.parameters()):,}")

### Open question: What if we respected the spatial structure?

We just flattened a 2D image into a 1D vector, discarding the spatial relationships between neighboring pixels. What if we used an operation that slides a small filter across the image and detects local patterns — edges, curves, corners?

**That is exactly what a Convolutional Neural Network (CNN) does.** CNNs achieve far better performance on image tasks and are the natural next step — but first, let us understand training with the simpler architecture.

### What you just learned

A PyTorch network is a Python class. You define the layers in `__init__` and the data flow in `forward`. The network starts with random weights and does nothing useful — training is what gives it meaning.

**Next question:** How do we train this network?

## 3. The Training Loop

A training loop repeats the same four steps for every batch, for every epoch:

| Step | What happens |
|---|---|
| **1. Forward pass** | Pass the batch through the network — get predictions |
| **2. Loss** | Measure how wrong the predictions are |
| **3. Backward pass** | Compute how each weight contributed to the error |
| **4. Weight update** | Adjust each weight to reduce the error |

We walk through each step on a single batch first, then assemble the full loop.

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS        = 25

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
batch_images, batch_labels = next(iter(train_loader))
batch_images = batch_images.to(device)
batch_labels = batch_labels.to(device)

print(f"Image batch shape: {batch_images.shape}  (batch x channels x height x width)")
print(f"Label batch shape: {batch_labels.shape}")

### Step 1: Forward Pass

Call the model like a function on the input batch. Data flows through:
`flatten` → `fc1` → `relu` → `fc2` → `relu` → `fc3`

The output is one vector of **10 numbers (logits)** per image — raw scores, one per class. They are not probabilities yet.

In [ ]:
logits = model(batch_images)

print(f"Output shape: {logits.shape}  (batch x 10 classes)")
print(f"Logits for first image: {logits[0].detach().cpu().numpy().round(2)}")

### Step 2: Loss Calculation

**Cross-Entropy Loss** is the standard choice for classification. It:
1. Applies softmax to convert logits to probabilities
2. Measures how unlikely the true class was under those probabilities

A random network on 10 classes produces a loss near `-log(1/10) ≈ 2.3`. Lower is better.

In [ ]:
loss = criterion(logits, batch_labels)
print(f"Loss: {loss.item():.4f}")

### Step 3: Backward Pass (Backpropagation)

`optimizer.zero_grad()` clears gradients from the previous step — without this, gradients would accumulate across batches.

`loss.backward()` uses the chain rule to compute, for each weight in the network, how much it contributed to the loss. After this call, each parameter `p` has `p.grad` filled in.

This gradient is the signal that makes learning possible.

In [ ]:
optimizer.zero_grad()
loss.backward()

print(f"Gradient norm of fc1.weight: {model.fc1.weight.grad.norm().item():.4f}")

### Step 4: Weight Update

`optimizer.step()` uses the gradients computed by `loss.backward()` to adjust each weight by a small amount in the direction that reduces the loss.

We use **Adam**, an adaptive optimizer that maintains per-parameter learning rates and converges faster than plain SGD on most deep learning tasks.

In [ ]:
optimizer.step()
print("Weights updated.")

### Putting it all together

Now we assemble the full loop over all epochs and all batches. A few additional details:

- `model.train()` activates training-mode behaviors (such as dropout, which we add in the next section)
- `model.eval()` + `torch.no_grad()` disables them during validation and skips gradient computation
- We record the average loss per epoch on both sets

In [ ]:
torch.manual_seed(42)
model     = FullyConnectedNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

train_losses = []
val_losses   = []

for epoch in range(1, EPOCHS + 1):

    # --- Training ---
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        logits = model(images)               # 1. forward pass
        loss   = criterion(logits, labels)   # 2. compute loss
        optimizer.zero_grad()                # 3a. clear old gradients
        loss.backward()                      # 3b. backward pass
        optimizer.step()                     # 4. update weights
        running += loss.item()

    # --- Validation ---
    model.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += criterion(model(images), labels).item()

    train_losses.append(running     / len(train_loader))
    val_losses.append(val_running / len(val_loader))

    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d}/{EPOCHS}  train={train_losses[-1]:.4f}  val={val_losses[-1]:.4f}")

### What you just learned

The training loop is the engine of deep learning. In PyTorch it is explicit: you see every step. The loop recorded train and validation loss at every epoch.

**Next question:** Did the model actually learn to generalize, or did it just memorize the training data?

## 4. Monitoring Training

We have two loss curves — one for training, one for validation. These are the primary diagnostic tool for understanding what a model learned.

In [ ]:
epochs_x = range(1, EPOCHS + 1)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(epochs_x, train_losses, color=COLOR_BASELINE, linewidth=2, label="Train loss")
ax.plot(epochs_x, val_losses,   color=COLOR_BASELINE, linewidth=2, linestyle="--", label="Validation loss")
ax.set_title("Training and validation loss — baseline model (no regularization)")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
plt.tight_layout()
plt.show()

### How to read these curves

- **Both curves decrease together:** the model is learning patterns that generalize.
- **Train loss keeps falling but validation loss plateaus or rises:** the model is *overfitting* — it has memorized the training set, not learned the underlying structure of digits.

The widening gap between the two curves is the visual signature of overfitting.

> **Contrast with scikit-learn:** `MLPClassifier` does not expose these curves. We could only detect overfitting by checking accuracy on a held-out set after training. Here, we can see it unfolding in real time, epoch by epoch.

### What you just learned

Loss curves diagnose overfitting in real time. The baseline model shows a growing gap between train and validation loss: it has more capacity than it needs and begins memorizing.

**Next question:** Can we keep the large network's capacity while preventing it from memorizing?

## 5. Regularization with Dropout

### What dropout does

During each forward pass in **training**, Dropout randomly sets a fraction `p` of activations to zero. A different random subset is zeroed on each batch.

This forces the network to learn **redundant representations** — no single neuron can become essential, because it might be zeroed out at any moment. The result: better generalization with the same architecture.

At inference time (`model.eval()`), dropout is turned off and all neurons are active. This is exactly why `model.train()` and `model.eval()` matter — they switch dropout on and off.

We add dropout after each hidden layer with `p=0.4` and retrain from scratch.

In [ ]:
class FullyConnectedNetDropout(nn.Module):
    def __init__(self, p=0.4):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1   = nn.Linear(784, 512)
        self.drop1 = nn.Dropout(p)
        self.fc2   = nn.Linear(512, 256)
        self.drop2 = nn.Dropout(p)
        self.fc3   = nn.Linear(256, 10)

    def forward(self, x):
        x = self.flatten(x)
        x = self.drop1(F.relu(self.fc1(x)))
        x = self.drop2(F.relu(self.fc2(x)))
        return self.fc3(x)

In [ ]:
torch.manual_seed(42)
model_do = FullyConnectedNetDropout().to(device)
optim_do = torch.optim.Adam(model_do.parameters(), lr=LEARNING_RATE)

train_losses_do = []
val_losses_do   = []

for epoch in range(1, EPOCHS + 1):

    model_do.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        loss = criterion(model_do(images), labels)
        optim_do.zero_grad()
        loss.backward()
        optim_do.step()
        running += loss.item()

    model_do.eval()
    val_running = 0.0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            val_running += criterion(model_do(images), labels).item()

    train_losses_do.append(running     / len(train_loader))
    val_losses_do.append(val_running / len(val_loader))

    if epoch % 5 == 0:
        print(f"Epoch {epoch:2d}/{EPOCHS}  train={train_losses_do[-1]:.4f}  val={val_losses_do[-1]:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(epochs_x, train_losses,    color=COLOR_BASELINE, linewidth=2, label="Baseline - train")
ax.plot(epochs_x, val_losses,      color=COLOR_BASELINE, linewidth=2, linestyle="--", label="Baseline - val")
ax.plot(epochs_x, train_losses_do, color=COLOR_DROPOUT,  linewidth=2, label="Dropout - train")
ax.plot(epochs_x, val_losses_do,   color=COLOR_DROPOUT,  linewidth=2, linestyle="--", label="Dropout - val")

ax.set_title("Effect of dropout on training and validation loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Cross-entropy loss")
ax.legend()
plt.tight_layout()
plt.show()

### What you just learned

Dropout is a structural change to the network — not a change to the training loop. The orange curves should show a smaller gap between train and validation loss compared to the blue curves. The dropout model generalizes better with the same number of parameters.

**Next question:** We used the validation set throughout to evaluate and compare models. How do we know the model actually generalizes to the real world?

## 6. Final Evaluation on the Test Set

### Why we waited until now

Throughout this notebook, the validation set was used to make implicit decisions:
- We observed that the baseline overfit, which led us to try dropout
- We compared the two models' validation curves to choose the better one
- We could have tuned the learning rate, architecture, or dropout rate to minimize validation loss

Every time we look at validation performance and make a decision, that information has influenced the model. Using the test set the same way would give an **optimistically biased** accuracy estimate.

**The test set is a sealed envelope.** We open it exactly once — right now — to get an unbiased estimate of real-world performance.

In [ ]:
best_model = model_do  # lower validation loss at convergence
best_model.eval()

all_preds, all_true, all_imgs = [], [], []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        preds  = best_model(images).argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_true.extend(labels.tolist())
        all_imgs.extend(images.cpu())

all_preds = np.array(all_preds)
all_true  = np.array(all_true)

test_acc = (all_preds == all_true).mean()
print(f"Test accuracy: {test_acc:.4f}  ({(all_preds == all_true).sum()}/{len(all_true)})")

In [ ]:
cm   = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=list(range(10)))

fig, ax = plt.subplots(figsize=(8, 7))
disp.plot(ax=ax, colorbar=True, cmap="Blues")
ax.set_title("Confusion Matrix — dropout model on test set")
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
plt.tight_layout()
plt.show()

In [ ]:
wrong_idx   = np.where(all_preds != all_true)[0]
correct_idx = np.where(all_preds == all_true)[0]
chosen      = np.concatenate([correct_idx[:8], wrong_idx[:8]])

all_imgs_t = torch.stack(all_imgs)

fig, axes = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axes.flat):
    idx  = chosen[i]
    img  = all_imgs_t[idx].squeeze() * 0.3081 + 0.1307
    true = all_true[idx]
    pred = all_preds[idx]
    ax.imshow(img, cmap="gray")
    color = "green" if true == pred else "red"
    ax.set_title(f"T:{true} P:{pred}", color=color, fontsize=10)
    ax.axis("off")
plt.suptitle("Test predictions — green=correct (top row), red=wrong (bottom row)", fontsize=12)
plt.tight_layout()
plt.show()

## Summary

This notebook traced the complete lifecycle of a deep learning project on image data.

### What we built
A fully connected neural network in PyTorch — with explicit layers, weights, and activation functions, rather than a black-box API.

### Four lessons

1. **The training loop** is four explicit steps: forward pass, loss, backward pass, weight update. PyTorch gives full control over each step.
2. **Loss curves** are the primary diagnostic tool. Overfitting is visible as a growing gap between train and validation loss.
3. **Dropout** It is a regularization technique. It randomly disables neurons during training so the neural network generalizes better to new data.
4. **The test set is a sealed envelope.** Every time you look at validation performance and make a decision, that information influences the model. Open the test set exactly once.

### What comes next

This network flattened each image from 28×28 to a 784-dimensional vector, discarding spatial relationships between neighboring pixels. A **Convolutional Neural Network (CNN)** respects this spatial structure using local filters that slide across the image — and achieves significantly higher accuracy with fewer parameters.